In [1]:
import pandas as pd
import numpy as np
import joblib
import os

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/processed/feature_engineered_data.csv")

df["Date"] = pd.to_datetime(df["Date"])

print("Dataset shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())

Dataset shape: (16440, 28)
Date range: 2022-01-31 00:00:00 to 2026-08-01 00:00:00


In [3]:
model = joblib.load("../models/demand_forecasting_model.pkl")
preprocessor = joblib.load("../models/preprocessor.pkl")

print("Demand forecasting model loaded successfully.")
print("Preprocessor loaded successfully.")

Demand forecasting model loaded successfully.
Preprocessor loaded successfully.


In [4]:
feature_columns = [
    "Product_ID",
    "Category",
    "Brand",
    "Current_Stock",
    "Unit_Price",
    "Discount_Percent",
    "Promotion",
    "Holiday",
    "Lead_Time_Days",
    "Safety_Stock",
    "Stockout",
    "Year",
    "Month",
    "Day",
    "Day_of_Week",
    "Week_of_Year",
    "Is_Weekend",
    "Lag_1_Day",
    "Lag_7_Days",
    "Lag_14_Days",
    "Rolling_7_Day",
    "Rolling_30_Day",
    "Rolling_7_Day_Std",
    "Discount_Amount",
    "Final_Price"
]

In [5]:
print("Number of features:", len(feature_columns))

Number of features: 25


In [6]:
X = df[feature_columns]

X_processed = preprocessor.transform(X)

print("Processed data shape:", X_processed.shape)

Processed data shape: (16440, 43)


In [7]:
df["Predicted_Demand"] = model.predict(X_processed)

df["Predicted_Demand"] = df["Predicted_Demand"].clip(lower=0)

df[[
    "Date",
    "Product_ID",
    "Product_Name",
    "Units_Sold",
    "Predicted_Demand"
]].head(10)

,Date,Product_ID,Product_Name,Units_Sold,Predicted_Demand
0,2022-01-31,P001,iPhone 13,3,7.791965
1,2022-02-01,P001,iPhone 13,8,7.135498
2,2022-02-02,P001,iPhone 13,8,7.437754
3,2022-02-03,P001,iPhone 13,6,7.184229
4,2022-02-04,P001,iPhone 13,5,7.439176
5,2022-02-05,P001,iPhone 13,6,7.580540
6,2022-02-06,P001,iPhone 13,8,7.938253
7,2022-02-07,P001,iPhone 13,6,7.214055
8,2022-02-08,P001,iPhone 13,9,7.336091
9,2022-02-09,P001,iPhone 13,5,7.140221


In [8]:
df[[
    "Product_Name",
    "Units_Sold",
    "Predicted_Demand"
]].head(10)

,Product_Name,Units_Sold,Predicted_Demand
0,iPhone 13,3,7.791965
1,iPhone 13,8,7.135498
2,iPhone 13,8,7.437754
3,iPhone 13,6,7.184229
4,iPhone 13,5,7.439176
5,iPhone 13,6,7.580540
6,iPhone 13,8,7.938253
7,iPhone 13,6,7.214055
8,iPhone 13,9,7.336091
9,iPhone 13,5,7.140221


demand variability

In [9]:
df[[
    "Product_Name",
    "Rolling_7_Day_Std",
    "Lead_Time_Days",
    "Safety_Stock"
]].head(10)

,Product_Name,Rolling_7_Day_Std,Lead_Time_Days,Safety_Stock
0,iPhone 13,2.544836,7.0,15
1,iPhone 13,2.935821,7.0,15
2,iPhone 13,2.768875,7.0,15
3,iPhone 13,2.751623,7.0,15
4,iPhone 13,2.751623,7.0,15
5,iPhone 13,2.853569,7.0,15
6,iPhone 13,2.853569,7.0,15
7,iPhone 13,1.889822,7.0,15
8,iPhone 13,1.253566,7.0,15
9,iPhone 13,1.463850,7.0,15


Dynamic Safety Stock=
Demand Standard Deviation
×
√(Lead Time)
×
Service Level Factor

In [10]:
service_level_z = 1.65

df["Dynamic_Safety_Stock"] = (
    df["Rolling_7_Day_Std"]
    * np.sqrt(df["Lead_Time_Days"])
    * service_level_z
)

In [11]:
df["Dynamic_Safety_Stock"] = np.ceil(
    df["Dynamic_Safety_Stock"]
)

In [12]:
df[[
    "Product_Name",
    "Rolling_7_Day_Std",
    "Lead_Time_Days",
    "Dynamic_Safety_Stock"
]].head(10)

,Product_Name,Rolling_7_Day_Std,Lead_Time_Days,Dynamic_Safety_Stock
0,iPhone 13,2.544836,7.0,12.0
1,iPhone 13,2.935821,7.0,13.0
2,iPhone 13,2.768875,7.0,13.0
3,iPhone 13,2.751623,7.0,13.0
4,iPhone 13,2.751623,7.0,13.0
5,iPhone 13,2.853569,7.0,13.0
6,iPhone 13,2.853569,7.0,13.0
7,iPhone 13,1.889822,7.0,9.0
8,iPhone 13,1.253566,7.0,6.0
9,iPhone 13,1.463850,7.0,7.0


Lead-Time Demand=
Predicted Daily Demand × Lead Time

In [13]:
df["Lead_Time_Demand"] = (
    df["Predicted_Demand"]
    * df["Lead_Time_Days"]
)

Dynamic Reorder Point=
Lead-Time Demand
+
Dynamic Safety Stock

In [14]:
df["Dynamic_Reorder_Point"] = (
    df["Lead_Time_Demand"]
    + df["Dynamic_Safety_Stock"]
)

In [15]:
df["Dynamic_Reorder_Point"] = np.ceil(
    df["Dynamic_Reorder_Point"]
)

In [18]:
#Round up:
df["Reorder_Status"] = np.where(
    df["Current_Stock"] <= df["Dynamic_Reorder_Point"],
    "REORDER",
    "SUFFICIENT STOCK"
)

In [17]:
df[[
    "Product_Name",
    "Current_Stock",
    "Predicted_Demand",
    "Lead_Time_Days",
    "Dynamic_Safety_Stock",
    "Dynamic_Reorder_Point",
    "Reorder_Status"
]].head(20)

,Product_Name,Current_Stock,Predicted_Demand,Lead_Time_Days,Dynamic_Safety_Stock,Dynamic_Reorder_Point,Reorder_Status
0,iPhone 13,139.0,7.791965,7.0,12.0,67.0,SUFFICIENT STOCK
1,iPhone 13,121.0,7.135498,7.0,13.0,63.0,SUFFICIENT STOCK
2,iPhone 13,164.0,7.437754,7.0,13.0,66.0,SUFFICIENT STOCK
3,iPhone 13,120.0,7.184229,7.0,13.0,64.0,SUFFICIENT STOCK
4,iPhone 13,157.0,7.439176,7.0,13.0,66.0,SUFFICIENT STOCK
5,iPhone 13,116.0,7.580540,7.0,13.0,67.0,SUFFICIENT STOCK
6,iPhone 13,169.0,7.938253,7.0,13.0,69.0,SUFFICIENT STOCK
7,iPhone 13,129.0,7.214055,7.0,9.0,60.0,SUFFICIENT STOCK
8,iPhone 13,145.0,7.336091,7.0,6.0,58.0,SUFFICIENT STOCK
9,iPhone 13,112.0,7.140221,7.0,7.0,57.0,SUFFICIENT STOCK


Calculate Recommended Order Quantity

In [19]:
df["Target_Stock"] = (
    df["Dynamic_Reorder_Point"]
    + df["Predicted_Demand"] * df["Lead_Time_Days"]
)

df["Recommended_Order_Quantity"] = (
    df["Target_Stock"] - df["Current_Stock"]
).clip(lower=0)

df["Recommended_Order_Quantity"] = np.ceil(
    df["Recommended_Order_Quantity"]
)

Only recommend an order when needed

In [20]:
df["Recommended_Order_Quantity"] = np.where(
    df["Reorder_Status"] == "REORDER",
    df["Recommended_Order_Quantity"],
    0
)

final inventory decision table

In [21]:
inventory_decisions = df[[
    "Date",
    "Product_ID",
    "Product_Name",
    "Category",
    "Current_Stock",
    "Predicted_Demand",
    "Lead_Time_Days",
    "Dynamic_Safety_Stock",
    "Dynamic_Reorder_Point",
    "Reorder_Status",
    "Recommended_Order_Quantity"
]].copy()

inventory_decisions.head(20)

,Date,Product_ID,Product_Name,Category,Current_Stock,Predicted_Demand,Lead_Time_Days,Dynamic_Safety_Stock,Dynamic_Reorder_Point,Reorder_Status,Recommended_Order_Quantity
0,2022-01-31,P001,iPhone 13,Mobile,139.0,7.791965,7.0,12.0,67.0,SUFFICIENT STOCK,0.0
1,2022-02-01,P001,iPhone 13,Mobile,121.0,7.135498,7.0,13.0,63.0,SUFFICIENT STOCK,0.0
2,2022-02-02,P001,iPhone 13,Mobile,164.0,7.437754,7.0,13.0,66.0,SUFFICIENT STOCK,0.0
3,2022-02-03,P001,iPhone 13,Mobile,120.0,7.184229,7.0,13.0,64.0,SUFFICIENT STOCK,0.0
4,2022-02-04,P001,iPhone 13,Mobile,157.0,7.439176,7.0,13.0,66.0,SUFFICIENT STOCK,0.0
5,2022-02-05,P001,iPhone 13,Mobile,116.0,7.580540,7.0,13.0,67.0,SUFFICIENT STOCK,0.0
6,2022-02-06,P001,iPhone 13,Mobile,169.0,7.938253,7.0,13.0,69.0,SUFFICIENT STOCK,0.0
7,2022-02-07,P001,iPhone 13,Mobile,129.0,7.214055,7.0,9.0,60.0,SUFFICIENT STOCK,0.0
8,2022-02-08,P001,iPhone 13,Mobile,145.0,7.336091,7.0,6.0,58.0,SUFFICIENT STOCK,0.0
9,2022-02-09,P001,iPhone 13,Mobile,112.0,7.140221,7.0,7.0,57.0,SUFFICIENT STOCK,0.0


Check how many reorder alerts we have

In [22]:
print(
    df["Reorder_Status"].value_counts()
)

Reorder_Status
SUFFICIENT STOCK    16108
REORDER               332
Name: count, dtype: int64


In [23]:
reorder_percentage = (
    (df["Reorder_Status"] == "REORDER").mean() * 100
)

print(f"Reorder rate: {reorder_percentage:.2f}%")

Reorder rate: 2.02%


See actual reorder alerts

In [25]:
reorder_alerts = inventory_decisions[
    inventory_decisions["Reorder_Status"] == "REORDER"
]

reorder_alerts.head(20)

,Date,Product_ID,Product_Name,Category,Current_Stock,Predicted_Demand,Lead_Time_Days,Dynamic_Safety_Stock,Dynamic_Reorder_Point,Reorder_Status,Recommended_Order_Quantity
48,2022-03-20,P001,iPhone 13,Mobile,2.0,4.082547,7.0,10.0,39.0,REORDER,66.0
104,2022-05-15,P001,iPhone 13,Mobile,1.0,3.670847,7.0,13.0,39.0,REORDER,64.0
339,2023-01-05,P001,iPhone 13,Mobile,2.0,3.097069,7.0,9.0,31.0,REORDER,51.0
480,2023-05-26,P001,iPhone 13,Mobile,2.0,3.332843,7.0,14.0,38.0,REORDER,60.0
541,2023-07-26,P001,iPhone 13,Mobile,0.0,3.091136,7.0,10.0,32.0,REORDER,54.0
645,2023-11-07,P001,iPhone 13,Mobile,2.0,3.093609,7.0,15.0,37.0,REORDER,57.0
666,2023-11-28,P001,iPhone 13,Mobile,1.0,3.191632,7.0,20.0,43.0,REORDER,65.0
752,2024-02-22,P001,iPhone 13,Mobile,0.0,3.481302,7.0,13.0,38.0,REORDER,63.0
824,2024-05-04,P001,iPhone 13,Mobile,1.0,3.923058,7.0,16.0,44.0,REORDER,71.0
842,2024-05-22,P001,iPhone 13,Mobile,0.0,3.281397,7.0,9.0,32.0,REORDER,55.0


Product-level summary

In [26]:
latest_inventory = (
    df.sort_values("Date")
      .groupby("Product_ID")
      .tail(1)
      .copy()
)

In [27]:
latest_inventory[[
    "Date",
    "Product_ID",
    "Product_Name",
    "Category",
    "Current_Stock",
    "Predicted_Demand",
    "Dynamic_Safety_Stock",
    "Dynamic_Reorder_Point",
    "Reorder_Status",
    "Recommended_Order_Quantity"
]]

,Date,Product_ID,Product_Name,Category,Current_Stock,Predicted_Demand,Dynamic_Safety_Stock,Dynamic_Reorder_Point,Reorder_Status,Recommended_Order_Quantity
8219,2026-08-01,P005,Dell Inspiron 15,Laptop,95.0,3.577366,8.0,44.0,SUFFICIENT STOCK,0.0
13151,2026-08-01,P008,Samsung 55-inch Smart TV,TV,110.0,3.153067,14.0,59.0,SUFFICIENT STOCK,0.0
11507,2026-08-01,P007,Lenovo IdeaPad 3,Laptop,117.0,4.182192,7.0,49.0,SUFFICIENT STOCK,0.0
9863,2026-08-01,P006,HP Pavilion 15,Laptop,126.0,5.021199,9.0,55.0,SUFFICIENT STOCK,0.0
6575,2026-08-01,P004,Redmi Note 13,Mobile,108.0,8.639481,12.0,56.0,SUFFICIENT STOCK,0.0
4931,2026-08-01,P003,Samsung Galaxy A55,Mobile,163.0,8.952414,10.0,64.0,SUFFICIENT STOCK,0.0
3287,2026-08-01,P002,iPhone 15,Mobile,148.0,7.325958,19.0,71.0,SUFFICIENT STOCK,0.0
1643,2026-08-01,P001,iPhone 13,Mobile,186.0,8.152972,16.0,74.0,SUFFICIENT STOCK,0.0
14795,2026-08-01,P009,LG 43-inch Smart TV,TV,112.0,3.156775,7.0,45.0,SUFFICIENT STOCK,0.0
16439,2026-08-01,P010,Sony 50-inch Bravia TV,TV,47.0,1.921249,8.0,35.0,SUFFICIENT STOCK,0.0


In [28]:
os.makedirs("../data/processed", exist_ok=True)

inventory_decisions.to_csv(
    "../data/processed/reorder_decisions.csv",
    index=False
)

latest_inventory.to_csv(
    "../data/processed/latest_inventory_status.csv",
    index=False
)

print("Reorder decisions saved successfully.")

Reorder decisions saved successfully.


In [30]:
reorder_alerts[[
    "Date",
    "Product_ID",
    "Product_Name",
    "Category",
    "Current_Stock",
    "Predicted_Demand",
    "Lead_Time_Days",
    "Dynamic_Safety_Stock",
    "Dynamic_Reorder_Point",
    "Reorder_Status",
    "Recommended_Order_Quantity"
]].head(20)

,Date,Product_ID,Product_Name,Category,Current_Stock,Predicted_Demand,Lead_Time_Days,Dynamic_Safety_Stock,Dynamic_Reorder_Point,Reorder_Status,Recommended_Order_Quantity
48,2022-03-20,P001,iPhone 13,Mobile,2.0,4.082547,7.0,10.0,39.0,REORDER,66.0
104,2022-05-15,P001,iPhone 13,Mobile,1.0,3.670847,7.0,13.0,39.0,REORDER,64.0
339,2023-01-05,P001,iPhone 13,Mobile,2.0,3.097069,7.0,9.0,31.0,REORDER,51.0
480,2023-05-26,P001,iPhone 13,Mobile,2.0,3.332843,7.0,14.0,38.0,REORDER,60.0
541,2023-07-26,P001,iPhone 13,Mobile,0.0,3.091136,7.0,10.0,32.0,REORDER,54.0
645,2023-11-07,P001,iPhone 13,Mobile,2.0,3.093609,7.0,15.0,37.0,REORDER,57.0
666,2023-11-28,P001,iPhone 13,Mobile,1.0,3.191632,7.0,20.0,43.0,REORDER,65.0
752,2024-02-22,P001,iPhone 13,Mobile,0.0,3.481302,7.0,13.0,38.0,REORDER,63.0
824,2024-05-04,P001,iPhone 13,Mobile,1.0,3.923058,7.0,16.0,44.0,REORDER,71.0
842,2024-05-22,P001,iPhone 13,Mobile,0.0,3.281397,7.0,9.0,32.0,REORDER,55.0


In [31]:
reorder_alerts["Product_Name"].value_counts()

Product_Name
Samsung 55-inch Smart TV    45
iPhone 15                   43
Lenovo IdeaPad 3            40
Samsung Galaxy A55          40
iPhone 13                   38
Redmi Note 13               33
LG 43-inch Smart TV         28
HP Pavilion 15              26
Dell Inspiron 15            22
Sony 50-inch Bravia TV      17
Name: count, dtype: int64

In [32]:
reorder_alerts["Category"].value_counts()

Category
Mobile    154
TV         90
Laptop     88
Name: count, dtype: int64

In [33]:
print("Total records:", len(df))
print("Reorder records:", (df["Reorder_Status"] == "REORDER").sum())
print("Sufficient stock:", (df["Reorder_Status"] == "SUFFICIENT STOCK").sum())

print(
    "Reorder percentage:",
    round((df["Reorder_Status"] == "REORDER").mean() * 100, 2),
    "%"
)

Total records: 16440
Reorder records: 332
Sufficient stock: 16108
Reorder percentage: 2.02 %
